# Problem Solver Tutor: 학습 문제 해결 에이전트

## 과제 주제
**교육 & 학습 분야 에이전트 설계 및 LangGraph 기초 구현**

이 노트북은 사용자가 자신의 학습 문제를 말하면, 그 문제를 분석하고 해결 계획과 연습 문제를 만들어 주는 **Problem Solver Tutor** 에이전트를 설계하고 구현합니다.


## Step 1. 에이전트 설계

### 1. 이름
**Problem Solver Tutor**

### 2. 목적
학습자가 막연하게 느끼는 학습 문제를 입력하면, 에이전트가 다음을 수행합니다.

1. 사용자의 문제를 분석한다.
2. 문제의 원인을 학습 관점에서 정리한다.
3. 해결 가능한 학습 목표로 바꾼다.
4. 단계별 학습 계획을 만든다.
5. 간단한 연습 문제와 피드백을 제공한다.

즉, 이 에이전트는 단순히 정답을 알려주는 챗봇이 아니라, 사용자의 문제를 **학습 가능한 과제**로 바꾸어 주는 맞춤형 학습 코치입니다.


## 해결하려는 문제

많은 학습자는 자신이 무엇 때문에 막히는지 정확히 모릅니다.

예를 들어:

- "영어 회화가 안 돼요."
- "코딩에서 함수가 너무 어려워요."
- "시험 공부를 해야 하는데 집중이 안 돼요."
- "책을 읽어도 금방 잊어버려요."

이런 문제는 겉으로는 단순한 고민처럼 보이지만, 실제로는 원인이 다를 수 있습니다.

예를 들어 영어 회화가 안 되는 이유도 다음처럼 나뉠 수 있습니다.

- 단어 부족
- 문장 패턴 부족
- 말하기 연습 부족
- 자신감 부족
- 실제 대화 상황 경험 부족

**Problem Solver Tutor**는 사용자의 문제를 분석해서 원인, 목표, 계획, 연습 문제로 바꾸어 줍니다.


## 핵심 기능

### 기능 1. 문제 분석
사용자가 말한 고민을 읽고, 핵심 문제와 가능한 원인을 정리합니다.

### 기능 2. 학습 목표 설정
막연한 고민을 실행 가능한 학습 목표로 바꿉니다.

예시:

> 영어를 잘하고 싶다  
> → 2주 안에 자기소개와 일상 대화를 1분 이상 말할 수 있다.

### 기능 3. 맞춤형 학습 계획 생성
사용자의 문제에 맞게 단계별 학습 계획을 제공합니다.

### 기능 4. 연습 문제 생성
학습 계획에 맞는 간단한 퀴즈, 연습 과제, 체크리스트를 만듭니다.

### 기능 5. 피드백 제공
사용자가 연습 답안을 입력하면 잘한 점과 보완할 점을 알려줍니다.


## 그래프 구조

아래는 Problem Solver Tutor의 전체 구조입니다.

```mermaid
flowchart TD
    START([START]) --> A[problem_analyzer]
    A --> B[goal_setter]
    B --> C[solution_planner]
    C --> D[practice_generator]
    D --> E[feedback_coach]
    E --> END([END])
```

### 노드 설명

| 노드 | 역할 |
|---|---|
| problem_analyzer | 사용자의 문제를 분석하고 핵심 원인을 정리 |
| goal_setter | 문제를 학습 목표로 변환 |
| solution_planner | 단계별 해결 계획 생성 |
| practice_generator | 연습 문제나 과제 생성 |
| feedback_coach | 최종 피드백 및 다음 행동 제안 |

이 과제의 요구사항은 최소 2개의 작동하는 노드입니다. 이 노트북에서는 더 완성도 있게 보이도록 5개의 노드를 구현합니다.


## Step 2. LangGraph 기초 구축

아래 코드는 LangGraph를 사용하여 기본 그래프를 구현합니다.

실행 환경에 LangGraph가 설치되어 있지 않다면 아래 설치 코드를 먼저 실행하세요.


In [ ]:

pip install -U langgraph typing_extensions

## 1. State 정의

 `LearningProblemState`라는 커스텀 State를 사용합니다.


In [ ]:
from typing_extensions import TypedDict
from typing import List
from langgraph.graph import StateGraph, START, END


class LearningProblemState(TypedDict):
    # 사용자가 입력한 원래 문제
    user_problem: str

    # 문제 분석 결과
    analysis: str

    # 학습 목표
    learning_goal: str

    # 해결 계획
    study_plan: List[str]

    # 연습 문제
    practice_tasks: List[str]

    # 최종 피드백
    feedback: str


## 2. 노드 구현

각 노드는 State를 입력받고, State의 일부를 업데이트해서 반환합니다.

이번 에이전트에서는 다음 5개의 노드를 구현합니다.

1. `problem_analyzer`
2. `goal_setter`
3. `solution_planner`
4. `practice_generator`
5. `feedback_coach`


In [ ]:
def problem_analyzer(state: LearningProblemState) -> dict:
    """사용자의 학습 문제를 분석하는 노드"""
    problem = state["user_problem"]

    analysis = f"""
사용자의 문제: {problem}

핵심 분석:
- 사용자는 현재 학습 과정에서 막힘을 느끼고 있습니다.
- 문제는 단순히 의지 부족이 아니라, 학습 방법·연습 구조·피드백 부족에서 발생했을 가능성이 있습니다.
- 따라서 이 문제는 '원인 파악 → 목표 설정 → 작은 단위의 실행 계획'으로 해결하는 것이 좋습니다.
"""

    return {"analysis": analysis.strip()}


def goal_setter(state: LearningProblemState) -> dict:
    """분석 결과를 바탕으로 학습 목표를 설정하는 노드"""
    problem = state["user_problem"]

    learning_goal = f"""
학습 목표:
'{problem}'라는 고민을 해결하기 위해, 앞으로 2주 동안 매일 20분씩 연습하여
현재 막히는 부분을 설명하고, 관련 문제를 스스로 3개 이상 해결할 수 있는 상태를 목표로 합니다.
"""

    return {"learning_goal": learning_goal.strip()}


def solution_planner(state: LearningProblemState) -> dict:
    """단계별 해결 계획을 만드는 노드"""
    study_plan = [
        "1단계: 현재 어려운 부분을 한 문장으로 정리한다.",
        "2단계: 문제의 원인을 지식 부족, 연습 부족, 집중 부족, 피드백 부족 중 어디에 가까운지 분류한다.",
        "3단계: 관련 개념을 아주 쉬운 예시 1개로 다시 학습한다.",
        "4단계: 작은 연습 문제를 3개 풀어 본다.",
        "5단계: 틀린 부분을 기록하고 다음 날 다시 복습한다.",
        "6단계: 1주일 후 같은 유형의 문제를 다시 풀어 성장 여부를 확인한다."
    ]

    return {"study_plan": study_plan}


def practice_generator(state: LearningProblemState) -> dict:
    """사용자 문제에 맞는 연습 과제를 생성하는 노드"""
    practice_tasks = [
        "연습 1: 내가 어려워하는 내용을 초등학생도 이해할 수 있게 3문장으로 설명해 보세요.",
        "연습 2: 관련 개념의 쉬운 예시를 하나 직접 만들어 보세요.",
        "연습 3: 오늘 배운 내용을 바탕으로 스스로 확인 문제 1개를 만들어 보세요.",
        "연습 4: 내일 다시 풀어볼 복습 질문을 하나 적어 두세요."
    ]

    return {"practice_tasks": practice_tasks}


def feedback_coach(state: LearningProblemState) -> dict:
    """최종 피드백을 제공하는 노드"""
    feedback = """
최종 피드백:
지금 사용자의 문제는 한 번에 해결하려고 하면 부담이 커질 수 있습니다.
따라서 문제를 작게 나누고, 매일 짧게 반복하는 방식이 가장 좋습니다.

추천 행동:
오늘은 먼저 '내가 정확히 어디에서 막히는지'를 한 문장으로 적어보세요.
그다음 가장 쉬운 예시 하나를 찾아 직접 설명해 보는 것부터 시작하면 됩니다.
"""

    return {"feedback": feedback.strip()}


## 3. 그래프 연결

이제 LangGraph의 `StateGraph`를 사용해 노드와 엣지를 연결합니다.


In [ ]:
graph_builder = StateGraph(LearningProblemState)

# 노드 추가
graph_builder.add_node("problem_analyzer", problem_analyzer)
graph_builder.add_node("goal_setter", goal_setter)
graph_builder.add_node("solution_planner", solution_planner)
graph_builder.add_node("practice_generator", practice_generator)
graph_builder.add_node("feedback_coach", feedback_coach)

# 엣지 연결
graph_builder.add_edge(START, "problem_analyzer")
graph_builder.add_edge("problem_analyzer", "goal_setter")
graph_builder.add_edge("goal_setter", "solution_planner")
graph_builder.add_edge("solution_planner", "practice_generator")
graph_builder.add_edge("practice_generator", "feedback_coach")
graph_builder.add_edge("feedback_coach", END)

# 그래프 컴파일
app = graph_builder.compile()


## 4. 테스트 실행

아래 예시는 사용자가 자신의 학습 문제를 입력했을 때 에이전트가 어떻게 작동하는지 보여줍니다.


In [ ]:
initial_state = {
    "user_problem": "코딩 공부를 시작했는데 함수 개념이 너무 어렵습니다.",
    "analysis": "",
    "learning_goal": "",
    "study_plan": [],
    "practice_tasks": [],
    "feedback": ""
}

result = app.invoke(initial_state)

result


## 5. 결과를 출력



In [ ]:
print("===== 사용자 문제 =====")
print(result["user_problem"])

print("\n===== 문제 분석 =====")
print(result["analysis"])

print("\n===== 학습 목표 =====")
print(result["learning_goal"])

print("\n===== 학습 계획 =====")
for item in result["study_plan"]:
    print("-", item)

print("\n===== 연습 과제 =====")
for task in result["practice_tasks"]:
    print("-", task)

print("\n===== 최종 피드백 =====")
print(result["feedback"])


## 6. 다른 입력 테스트




In [ ]:
test_problems = [
    "영어 단어는 외우는데 회화가 잘 안 됩니다.",
    "시험 공부를 해야 하는데 집중력이 너무 떨어집니다.",
    "책을 읽어도 내용을 금방 잊어버립니다.",
    "피아노를 배우고 있는데 악보 읽기가 너무 어렵습니다."
]

for problem in test_problems:
    output = app.invoke({
        "user_problem": problem,
        "analysis": "",
        "learning_goal": "",
        "study_plan": [],
        "practice_tasks": [],
        "feedback": ""
    })

    print("\n" + "=" * 80)
    print("입력 문제:", problem)
    print("-" * 80)
    print(output["analysis"])
    print("\n학습 목표:")
    print(output["learning_goal"])
    print("\n첫 번째 추천 계획:")
    print(output["study_plan"][0])
    print("\n최종 피드백:")
    print(output["feedback"])
